# ETL MV Batch CSV 测试流程

这个 notebook 每次运行都会创建新的 `run_id`。它复用 `llm_demo/artifacts/20260602_112546/01_query_blocks` 的 Feature 提取结果和 `02_families` 的 QueryFamily 结果，不重新运行 `FeatureAgent`、`FamilyAgent` 或 `BatchClusterAgent` 的 LLM 分类流程。

准备阶段会从 `batch_classification.csv` 写入当前 run 的 `03_batches/complexity_batches.json`，之后只针对你选择的 batch 运行 batch 内 MV 生成和 rewrite dry-run 闭环。


In [1]:
from pathlib import Path
import json
import shutil
import sys
from collections import Counter
from datetime import datetime
from dotenv import load_dotenv

# 如果本地刚修改过 llm_demo 源码，自动 reload 已导入模块。
%load_ext autoreload
%autoreload 2

# 自动定位项目根目录，并加入 Python import 路径。
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    path for path in [cwd, *cwd.parents]
    if (path / "llm_demo").is_dir() and (path / "tpcds-spark").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")
PROJECT_ROOT


PosixPath('/Users/zlhmac/code/Python_project/MV_gen')

In [2]:
from llm_demo.src.agents.batch_cluster_agent import BatchClusterAgent
from llm_demo.src.agents.batch_mv_agent import BatchMVAgent
from llm_demo.src.agents.executor_agent import ExecutorAgent
from llm_demo.src.agents.rewrite_agent import RewriteAgent
from llm_demo.src.agents.self_iteration_agent import SelfIterationAgent
from llm_demo.src.core.artifact_store import ArtifactStore
from llm_demo.src.core.coverage_summary_builder import CoverageSummaryBuilder
from llm_demo.src.core.llm_client import LLMClient


In [3]:
SOURCE_RUN_ID = "20260602_112546"
BATCH_CLASSIFICATION_CSV = PROJECT_ROOT / "batch_classification.csv"

# 每次实验使用独立 run_id；失败排查通过当前 run 的 artifacts 和 run_log 查看。
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
artifact_root = PROJECT_ROOT / "llm_demo" / "artifacts"
source_store = ArtifactStore(run_id=SOURCE_RUN_ID, artifact_root=artifact_root)
store = ArtifactStore(run_id=run_id, artifact_root=artifact_root)
llm_client = LLMClient(project_root=PROJECT_ROOT)

run_id, store.run_dir


('20260607_163234',
 PosixPath('/Users/zlhmac/code/Python_project/MV_gen/llm_demo/artifacts/20260607_163234'))

In [4]:
# 1. 复用旧 run 的 SQL manifest 和 Feature artifacts。
#    SQL manifest 是 rewrite 阶段读取原始 SQL 必需的输入；QueryBlock 相关文件来自旧 Feature 提取结果。
FEATURE_REUSE_PATHS = [
    "00_raw_sql/sql_manifest.json",
    "01_query_blocks/query_blocks.json",
    "01_query_blocks/query_to_qbs.json",
    "01_query_blocks/qb_to_query.json",
    "01_query_blocks/feature_extract_status.json",
]


def copy_artifact(relative_path):
    source_path = source_store.path(relative_path)
    target_path = store.path(relative_path)
    if not source_path.exists():
        raise FileNotFoundError(source_path)
    target_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_path, target_path)
    return target_path


copied_paths = {relative_path: copy_artifact(relative_path) for relative_path in FEATURE_REUSE_PATHS}

# 让当前 run 的 status artifact 与新 run_id 对齐，同时保留来源。
feature_status = store.read_json(store.feature_status_path)
feature_status["run_id"] = store.run_id
feature_status["source_run_id"] = SOURCE_RUN_ID
store.write_json("01_query_blocks/feature_extract_status.json", feature_status)

store.append_run_log(
    agent_name="NotebookSeed",
    event="reuse_feature_artifacts",
    input_artifact_paths=[source_store.path(relative_path) for relative_path in FEATURE_REUSE_PATHS],
    output_artifact_paths=list(copied_paths.values()),
    elapsed_ms=0,
    details={"source_run_id": SOURCE_RUN_ID},
)

copied_paths


{'00_raw_sql/sql_manifest.json': PosixPath('/Users/zlhmac/code/Python_project/MV_gen/llm_demo/artifacts/20260607_163234/00_raw_sql/sql_manifest.json'),
 '01_query_blocks/query_blocks.json': PosixPath('/Users/zlhmac/code/Python_project/MV_gen/llm_demo/artifacts/20260607_163234/01_query_blocks/query_blocks.json'),
 '01_query_blocks/query_to_qbs.json': PosixPath('/Users/zlhmac/code/Python_project/MV_gen/llm_demo/artifacts/20260607_163234/01_query_blocks/query_to_qbs.json'),
 '01_query_blocks/qb_to_query.json': PosixPath('/Users/zlhmac/code/Python_project/MV_gen/llm_demo/artifacts/20260607_163234/01_query_blocks/qb_to_query.json'),
 '01_query_blocks/feature_extract_status.json': PosixPath('/Users/zlhmac/code/Python_project/MV_gen/llm_demo/artifacts/20260607_163234/01_query_blocks/feature_extract_status.json')}

In [5]:
# 2. 不跑 FamilyAgent；复用旧 run 的 QueryFamily artifacts。
#    旧 families 与旧 Feature artifacts 来自同一个 SOURCE_RUN_ID，qb_id/member 边界保持一致。
FAMILY_REUSE_PATHS = [
    "02_families/family_candidates.json",
    "02_families/query_families.json",
]

copied_family_paths = {relative_path: copy_artifact(relative_path) for relative_path in FAMILY_REUSE_PATHS}
query_families = store.read_json(store.query_families_path)["query_families"]
family_candidates = store.read_json(store.family_candidates_path)["family_candidates"]

store.append_run_log(
    agent_name="NotebookSeed",
    event="reuse_family_artifacts",
    input_artifact_paths=[source_store.path(relative_path) for relative_path in FAMILY_REUSE_PATHS],
    output_artifact_paths=list(copied_family_paths.values()),
    elapsed_ms=0,
    details={
        "source_run_id": SOURCE_RUN_ID,
        "family_count": len(query_families),
        "family_candidate_count": len(family_candidates),
    },
)

{
    "families_path": store.query_families_path,
    "family_count": len(query_families),
    "family_candidates_path": store.family_candidates_path,
    "family_candidate_count": len(family_candidates),
}


{'families_path': PosixPath('/Users/zlhmac/code/Python_project/MV_gen/llm_demo/artifacts/20260607_163234/02_families/query_families.json'),
 'family_count': 96,
 'family_candidates_path': PosixPath('/Users/zlhmac/code/Python_project/MV_gen/llm_demo/artifacts/20260607_163234/02_families/family_candidates.json'),
 'family_candidate_count': 123}

In [6]:
# 3. 使用 batch_classification.csv 写入当前 run 的 ComplexityBatch artifact。
#    batch_classification.csv 是本轮 batch 划分的唯一来源；batch_type 直接写 Batch-1..Batch-5/Other。
batches_path = BatchClusterAgent(store, llm_client).run_from_classification_csv(
    classification_csv_path=BATCH_CLASSIFICATION_CSV,
    sql_manifest_path=store.sql_manifest_path,
    query_blocks_path=store.query_blocks_path,
    families_path=store.query_families_path,
)

complexity_batches = store.read_json(batches_path)["complexity_batches"]
last_batch_record = [
    json.loads(line)
    for line in store.run_log_path.read_text(encoding="utf-8").splitlines()
    if line.strip() and json.loads(line).get("agent_name") == "BatchClusterAgent"
][-1]

{
    "batches_path": batches_path,
    "batch_counts": {batch["batch_type"]: len(batch["query_ids"]) for batch in complexity_batches},
    "missing_feature_query_ids": last_batch_record["details"]["missing_feature_query_ids"],
    "sqlglot_parsed_query_count": last_batch_record["details"]["sqlglot_parsed_query_count"],
}


{'batches_path': PosixPath('/Users/zlhmac/code/Python_project/MV_gen/llm_demo/artifacts/20260607_163234/03_batches/complexity_batches.json'),
 'batch_counts': {'Batch-1': 15,
  'Batch-2': 29,
  'Batch-3': 16,
  'Batch-4': 17,
  'Batch-5': 27,
  'Other': 1},
 'missing_feature_query_ids': [],
 'sqlglot_parsed_query_count': 105}

In [7]:
# 4. 查看本轮准备好的 batch。若 missing_feature_query_ids 非空，对应 SQL 会保留在 query_ids 里，
#    但因为旧 Feature artifact 没有可用 QueryBlock，它们通常只能走 rewrite fallback，不适合作为 MV source。
batches = store.read_json(store.complexity_batches_path)["complexity_batches"]
feature_status = store.read_json(store.feature_status_path)
status_by_query = {item["query_id"]: item for item in feature_status["queries"]}
query_to_qbs = store.read_json(store.query_to_qbs_path)

batch_preview = [
    {
        "batch_id": batch["batch_id"],
        "batch_type": batch["batch_type"],
        "query_count": len(batch["query_ids"]),
        "family_group_count": len(batch.get("family_groups", [])),
        "missing_feature_query_ids": [query_id for query_id in batch["query_ids"] if query_id not in query_to_qbs],
        "unsupported_feature_query_ids": [
            query_id
            for query_id in batch["query_ids"]
            if status_by_query.get(query_id, {}).get("status") == "unsupported_sql_pattern"
        ],
    }
    for batch in batches
]

batch_preview


[{'batch_id': 1,
  'batch_type': 'Batch-1',
  'query_count': 15,
  'family_group_count': 11,
  'missing_feature_query_ids': [],
  'unsupported_feature_query_ids': []},
 {'batch_id': 2,
  'batch_type': 'Batch-2',
  'query_count': 29,
  'family_group_count': 28,
  'missing_feature_query_ids': [],
  'unsupported_feature_query_ids': []},
 {'batch_id': 3,
  'batch_type': 'Batch-3',
  'query_count': 16,
  'family_group_count': 16,
  'missing_feature_query_ids': [],
  'unsupported_feature_query_ids': ['q85']},
 {'batch_id': 4,
  'batch_type': 'Batch-4',
  'query_count': 17,
  'family_group_count': 28,
  'missing_feature_query_ids': [],
  'unsupported_feature_query_ids': []},
 {'batch_id': 5,
  'batch_type': 'Batch-5',
  'query_count': 27,
  'family_group_count': 40,
  'missing_feature_query_ids': [],
  'unsupported_feature_query_ids': ['q72']},
 {'batch_id': 6,
  'batch_type': 'Other',
  'query_count': 1,
  'family_group_count': 1,
  'missing_feature_query_ids': [],
  'unsupported_feature_que

In [8]:
# 5. 定义单 batch dry-run helper：historical rewrite -> batch-local MV -> materialize state -> final rewrite -> query order。
#    注意：这个 helper 会调用 RewriteAgent 和 BatchMVAgent 的 LLM；不会运行 FeatureAgent、FamilyAgent、BatchClusterAgent。

sql_manifest_path = store.sql_manifest_path
query_blocks_path = store.query_blocks_path
families_path = store.query_families_path
batches_path = store.complexity_batches_path


def find_batch(batch_id):
    for batch in store.read_json(batches_path)["complexity_batches"]:
        if batch["batch_id"] == batch_id:
            return batch
    raise ValueError(f"Batch {batch_id} not found")


def run_dry_run_batch(batch_id):
    batch = find_batch(batch_id)
    if not batch.get("query_ids"):
        raise ValueError(f"Batch {batch_id} has no query_ids")

    execution_order_path = store.execution_order_path(batch_id)
    if execution_order_path.exists():
        print(
            f"提示：batch {batch_id} 已存在 execution_order。"
            "如果要干净重跑，建议重新执行开头 cell 创建新 run_id。"
        )

    rewrite_agent = RewriteAgent(store, llm_client)
    batch_mv_agent = BatchMVAgent(store, llm_client)
    executor_agent = ExecutorAgent(store)
    mv_state_path = store.materialized_mvs_path

    historical_rewrite_dir = rewrite_agent.run(
        batch_id=batch_id,
        rewrite_stage="historical",
        complexity_batches_path=batches_path,
        sql_manifest_path=sql_manifest_path,
        query_blocks_path=query_blocks_path,
        materialized_mvs_path=mv_state_path,
    )

    mv_candidates_path = batch_mv_agent.run(
        batch_id=batch_id,
        complexity_batches_path=batches_path,
        query_blocks_path=query_blocks_path,
        families_path=families_path,
        historical_rewrite_dir=historical_rewrite_dir,
        materialized_mvs_path=mv_state_path,
    )

    mv_state_path = executor_agent.materialize_mvs(
        batch_id=batch_id,
        mv_candidates_path=mv_candidates_path,
        mv_build_sql_path=store.batch_mv_build_sql_path(batch_id),
        materialized_mvs_path=mv_state_path,
    )

    final_rewrite_dir = rewrite_agent.run(
        batch_id=batch_id,
        rewrite_stage="final",
        complexity_batches_path=batches_path,
        sql_manifest_path=sql_manifest_path,
        query_blocks_path=query_blocks_path,
        materialized_mvs_path=mv_state_path,
    )

    execution_order_path = executor_agent.run_queries(
        batch_id=batch_id,
        final_rewrite_dir=final_rewrite_dir,
        complexity_batches_path=batches_path,
    )

    return {
        "batch_id": batch_id,
        "query_ids": batch["query_ids"],
        "historical_rewrite_dir": historical_rewrite_dir,
        "mv_candidates_path": mv_candidates_path,
        "materialized_mvs_path": mv_state_path,
        "final_rewrite_dir": final_rewrite_dir,
        "execution_order_path": execution_order_path,
    }


def summarize_batch_outputs(outputs):
    mv_candidates = store.read_json(outputs["mv_candidates_path"])["mv_candidates"]
    execution_order = store.read_json(outputs["execution_order_path"])
    materialized_mvs = store.read_json(outputs["materialized_mvs_path"])["materialized_mvs"]

    final_meta_paths = sorted(outputs["final_rewrite_dir"].glob("*_rewrite_meta.json"))
    rewrite_status_counts = Counter(
        store.read_json(meta_path).get("status", "unknown")
        for meta_path in final_meta_paths
    )
    mv_decision_counts = Counter(candidate.get("decision", "unknown") for candidate in mv_candidates)

    return {
        "batch_id": outputs["batch_id"],
        "batch_type": find_batch(outputs["batch_id"])["batch_type"],
        "query_count": len(outputs["query_ids"]),
        "query_ids": outputs["query_ids"],
        "mv_candidate_count": len(mv_candidates),
        "mv_decision_counts": dict(mv_decision_counts),
        "materialized_mv_count": len(materialized_mvs),
        "rewrite_status_counts": dict(rewrite_status_counts),
        "execution_step_count": len(execution_order.get("steps", [])),
    }


In [9]:
# 6. 按需选择要测试的 batch。建议先小范围运行，例如 [1] 或 [1, 2]。
batch_ids_to_run = [1,2,3,4,5]

batch_outputs = [run_dry_run_batch(batch_id) for batch_id in batch_ids_to_run]
[summarize_batch_outputs(outputs) for outputs in batch_outputs]


[{'batch_id': 1,
  'batch_type': 'Batch-1',
  'query_count': 15,
  'query_ids': ['q01',
   'q02',
   'q15',
   'q19',
   'q22',
   'q26',
   'q3',
   'q42',
   'q43',
   'q52',
   'q55',
   'q62',
   'q7',
   'q96',
   'q99'],
  'mv_candidate_count': 11,
  'mv_decision_counts': {'materialize': 6, 'skip': 5},
  'materialized_mv_count': 27,
  'rewrite_status_counts': {'fallback': 7, 'rewritten': 8},
  'execution_step_count': 26},
 {'batch_id': 2,
  'batch_type': 'Batch-2',
  'query_count': 29,
  'query_ids': ['q12',
   'q13',
   'q18',
   'q20',
   'q21',
   'q27',
   'q28',
   'q32',
   'q34',
   'q39a',
   'q39b',
   'q45',
   'q46',
   'q48',
   'q53',
   'q59',
   'q6',
   'q61',
   'q63',
   'q65',
   'q68',
   'q73',
   'q79',
   'q88',
   'q89',
   'q9',
   'q90',
   'q92',
   'q98'],
  'mv_candidate_count': 7,
  'mv_decision_counts': {'materialize': 7},
  'materialized_mv_count': 27,
  'rewrite_status_counts': {'fallback': 28, 'rewritten': 1},
  'execution_step_count': 36},
 {'ba

In [10]:
# 7. 可选：生成 coverage summary。跑完至少一个 batch 后再执行。
coverage_path = CoverageSummaryBuilder(store).run()
store.read_json(coverage_path)


{'run_id': '20260607_163234',
 'total_queries': 105,
 'feature_status_counts': {'success': 70,
  'partial_success': 33,
  'unsupported_sql_pattern': 2},
 'family_candidate_count': 123,
 'family_count': 96,
 'batch_query_counts': {'1': 15, '2': 29, '3': 16, '4': 17, '5': 27, '6': 1},
 'mv_candidate_counts': {'materialize': 27, 'skip': 5},
 'rewrite_status_counts': {'final:fallback': 95,
  'final:rewritten': 9,
  'historical:fallback': 104},
 'execution_step_counts': {'materialize_mv:success': 27,
  'materialize_mv:skipped': 5,
  'run_query:planned': 104},
 'notes': ['feature_failed_or_unsupported_queries=2']}

In [11]:
# 8. 可选：生成 SelfIterationAgent 反馈。建议跑完目标 batch 后再执行。
feedback_path = SelfIterationAgent(store, llm_client).run(store.run_log_path)
store.read_json(feedback_path)


{'run_id': '20260607_163234',
 'agent_rule_suggestions': {'BatchMVAgent': {'suggestions': [{'target_rule': 'derived_expression_column_mapping_not_supported',
     'suggestion': 'clarify',
     'suggested_rule_text': '当 MV 候选包含 sum(CASE WHEN ...) 等派生表达式时，应尝试将其拆分为多个列或使用表达式别名，而不是直接 skip。例如，对于 sum(CASE WHEN condition THEN col ELSE 0 END)，可以在 build_sql 中保留该表达式并赋予显式别名，同时记录 column_mappings 中 source_expr 为完整表达式，mv_column 为别名。',
     'reason': '多个 MV 候选因 derived_expression_column_mapping_not_supported 被 skip，如 q43、q62、q99 的候选。',
     'evidence_refs': [{'artifact': None,
       'event_id': '20260607_163234:BatchMVAgent:batch_1:0028',
       'batch_id': 1,
       'candidate_id': 'cand_batch_1_family_store_sales_date_dim_store_0001',
       'mv_id': None,
       'query_ids': [],
       'event': 'mv_candidate_skipped'},
      {'artifact': None,
       'event_id': '20260607_163234:BatchMVAgent:batch_1:0029',
       'batch_id': 1,
       'candidate_id': 'cand_batch_1_family_web_sales_date_dim_ship_mo